In [ ]:
# ---------------- MOUNT DRIVE ----------------
from google.colab import drive
drive.mount('/content/drive')

# ---------------- IMPORTS ----------------
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image
from google.colab import files

# ---------------- MODEL PATH ----------------
MODEL_PATH = "/content/drive/MyDrive/Chatbot/soil_models/soil_model_epoch_8.pth"

# ---------------- CLASS LABELS ----------------
classes = [
    "Alluvial_Soil",
    "Arid_Soil",
    "Black_Soil",
    "Laterite_Soil",
    "Mountain_Soil",
    "Red_Soil",
    "Yellow_Soil"
]

# ---------------- TRANSFORM ----------------
transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.255])
])

# ---------------- ENSEMBLE MODEL ----------------
class Ensemble(nn.Module):

    def __init__(self,model1,model2,num_classes,f1,f2):
        super().__init__()

        self.model1 = model1
        self.model2 = model2

        self.model1.classifier[1] = nn.Identity()
        self.model2.heads[0] = nn.Identity()

        self.classifier = nn.Sequential(
            nn.Linear(f1 + f2, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes)
        )

        self.resize224 = transforms.Resize((224,224))

    def forward(self,x):

        x1 = self.model1(x)
        x1 = x1.view(x1.size(0),-1)

        x2 = torch.stack([self.resize224(img) for img in x])
        x2 = self.model2(x2)
        x2 = x2.view(x2.size(0),-1)

        x = torch.cat((x1,x2),dim=1)
        x = self.classifier(x)

        return x


# ---------------- LOAD MODELS ----------------
model1 = models.efficientnet_v2_l(weights=None)
f1 = model1.classifier[1].in_features
model1.classifier[1] = nn.Linear(f1,len(classes))

model2 = models.vit_l_16(weights=None)
f2 = model2.heads[0].in_features
model2.heads[0] = nn.Linear(f2,len(classes))

model = Ensemble(model1,model2,len(classes),f1,f2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.load_state_dict(torch.load(MODEL_PATH,map_location=device))
model.to(device)
model.eval()

print("Model loaded successfully")

# ---------------- UPLOAD IMAGE ----------------
uploaded = files.upload()
image_path = list(uploaded.keys())[0]

# ---------------- PREDICTION ----------------
img = Image.open(image_path).convert("RGB")
img = transform(img).unsqueeze(0).to(device)

with torch.no_grad():
    output = model(img)
    _, pred = torch.max(output,1)

print("Predicted Soil Type:", classes[pred.item()])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model loaded successfully


Saving Screenshot 2026-03-06 200753.png to Screenshot 2026-03-06 200753.png
Predicted Soil Type: Alluvial_Soil
